## Model Integration - Open AI, Google Gemini, Groq, Ollama, GE Models

LangChain can talk to many model providers through a single, consistent
pattern. Two entry points are used in this guide:

| Approach | When to use |
|----------|-------------|
| `init_chat_model("provider:model", ...)` | Uniform one-liner across providers |
| `ChatOpenAI(base_url=..., api_key=...)` | Any OpenAI-compatible endpoint (e.g., GE gateway) |

> **Key idea:** `init_chat_model` is a factory. With `model_provider` set, it
> builds the correct provider class under the hood and forwards extra kwargs
> (`base_url`, `api_key`, `temperature`, `timeout`, …) straight through.

---


In [20]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["OLLAMA_API_KEY"] = os.getenv("OLLAMA_API_KEY")

#### Open AI

In [21]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="gpt-4o",
)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000156D9D9C1A0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000156D9C77E60>, root_client=<openai.OpenAI object at 0x

In [ ]:
response = model.invoke("What is Langchain")
response

#### Google

In [ ]:
model = init_chat_model("google_genai:gemini-2.5-flash-lite")
# We need to add google_genai as prefix to model name to use the Google Gemini model. 

#### Ollama local model 

In [ ]:
model = init_chat_model("ollama:qwen3:8b")

In [ ]:
model.invoke("Hello")

AIMessage(content="Hello! 😊 How can I assist you today? I'm here to help with any questions or tasks you might have!", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-07-30T13:34:45.7125123Z', 'done': True, 'done_reason': 'stop', 'total_duration': 39160275400, 'load_duration': 811919000, 'prompt_eval_count': 11, 'prompt_eval_duration': 3431314000, 'eval_count': 108, 'eval_duration': 34572827000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019fb33b-75b8-7651-ac06-ee80ac220470-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 108, 'total_tokens': 119})

### GE Models

In [37]:

ge_model = init_chat_model(
    "llama3.2",                          # approved: gemma3:27b, llama3.2, gpt-oss...
    model_provider="openai",             # gateway speaks the OpenAI API (LiteLLM)
    base_url=os.getenv("GE_BASE_URL"),  # gateway URL
    api_key=os.getenv("GE_API_KEY"),    # gateway API key
    temperature=0,
    timeout=30,
)
print(ge_model.invoke("Hello from GE ").content)


Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


## Streaming and Batch

## Streaming and Batch

LangChain models and chains support multiple execution modes beyond a single
`invoke()`. The main ones are **streaming** (get output token-by-token) and
**batch** (process many inputs efficiently).

### Streaming

Streaming yields the response incrementally as it's generated, instead of
waiting for the full reply. This is ideal for chat UIs and long outputs.

```python
for chunk in model.stream("Explain what LangChain is in 3 sentences"):
    print(chunk.content, end="", flush=True)
```

- Each `chunk` is a partial message (`AIMessageChunk`).
- `end=""` + `flush=True` print tokens smoothly on one line.
- Works the same for models, chains, and agents.

**Streaming a chain:**

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Write a short note about {topic}")
chain = prompt | model | StrOutputParser()

for chunk in chain.stream({"topic": "React hooks"}):
    print(chunk, end="", flush=True)
```

**Async streaming** (for async apps / notebooks):

```python
async for chunk in model.astream("Tell me a fact about the sun"):
    print(chunk.content, end="", flush=True)
```
### Notes for the GE gateway

- Streaming and batch both work through the OpenAI-compatible GE gateway
  (`ChatOpenAI` / `model_provider="openai"`).
- For **batch**, set a conservative `max_concurrency` to respect gateway rate
  limits.
- If streaming hangs, confirm VPN connectivity and that the corporate SSL
  certificate is trusted (see the Troubleshooting section).

In [38]:
# Streaming

model.stream("Write me a 200 words essay on the importance of AI in helathcare")

<generator object BaseChatModel.stream at 0x00000156D9DCE510>

In [ ]:
import os, httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

http_client = httpx.Client(verify=False)          # ⚠️ dev only

model = ChatOpenAI(                                # 👈 REBUILD model here
    model="llama3.2",
    base_url=os.getenv("GE_BASE_URL"),
    api_key=os.getenv("GE_API_KEY"),
    temperature=0,
    http_client=http_client,                       # 👈 attach the client
)
## Notes : As the content is generated from LLM, It is steamed in chunks, and LLM is also giving the output
# Not waiting for complete output to be generated and then log it.
for chunk in model.stream("Write me a 200-word essay on AI in healthcare"):
    print(chunk.text)       # 👈 .content, not .text

The
 integration
 of
 Artificial
 Intelligence
 (
AI
)
 in
 healthcare
 has
 revolution
ized
 the
 way
 medical
 professionals
 diagnose
,
 treat
,
 and
 prevent
 diseases
.
 AI
 algorithms
 can
 analyze
 vast
 amounts
 of
 patient
 data
,
 identify
 patterns
,
 and
 make
 predictions
 that
 would
 be
 impossible
 for
 humans
 to
 achieve
 alone
.


In
 diagnostic
 medicine
,
 AI
-powered
 systems
 can
 help
 doctors
 detect
 diseases
 such
 as
 cancer
,
 diabetes
,
 and
 cardiovascular
 disease
 more
 accurately
 and
 quickly
 than
 traditional
 methods
.
 For
 example
,
 AI
-driven
 computer
 vision
 can
 analyze
 medical
 images
 like
 X
-rays
 and
 MR
Is
 to
 detect
 abnormalities
 and
 identify
 potential
 health
 risks
.


AI
 is
 also
 being
 used
 in
 personalized
 medicine
 to
 tailor
 treatment
 plans
 to
 individual
 patients
 based
 on
 their
 genetic
 profiles
,
 medical
 histories
,
 and
 lifestyle
 factors
.
 Additionally
,
 AI
-powered
 chat
bots
 and
 virtual
 assistan

## `end` and `flush` in `print()` (Streaming)

### `end`
Controls what is added **after** each printed chunk.

- Default is `"\n"` (newline) → each chunk goes on a new line.
- Use `end=""` → chunks join into one smooth response.
- Use `end=" | "` → shows the gap between chunks (good for debugging).

```python
print(chunk.content, end="", flush=True)      # smooth text
print(chunk.content, end=" | ", flush=True)   # see each chunk
```

### `flush`
Controls **when** text appears.

- Default is `False` → Python may hold text and print in bursts.
- Use `flush=True` → each chunk shows instantly (live typing effect).

### Quick table

| Setting | Default | Best for streaming | Why |
|---------|---------|--------------------|-----|
| `end` | `"\n"` | `""` | Keeps text on one flow |
| `flush` | `False` | `True` | Shows each chunk right away |

### Note
Use `chunk.content` (not `.text`) for chat models.

In [44]:
## Better way to display the streanmed content
for chunk in model.stream("Write me a 200-word essay on AI in healthcare"):
    print(chunk.text, end=" | ", flush=True)       # 👈 .content, not .text

The |  integration |  of |  Artificial |  Intelligence |  ( | AI | ) |  in |  healthcare |  has |  revolution | ized |  the |  way |  medical |  professionals |  diagnose | , |  treat | , |  and |  prevent |  diseases | . |  AI |  algorithms |  can |  analyze |  vast |  amounts |  of |  patient |  data | , |  identify |  patterns | , |  and |  make |  predictions |  that |  would |  be |  impossible |  for |  humans |  to |  achieve |  alone | .

 | In |  diagnostic |  medicine | , |  AI | -powered |  systems |  can |  help |  doctors |  detect |  diseases |  such |  as |  cancer | , |  diabetes | , |  and |  cardiovascular |  disease |  more |  accurately |  and |  quickly |  than |  traditional |  methods | . |  For |  example | , |  AI | -driven |  computer |  vision |  can |  analyze |  medical |  images |  like |  X | -rays |  and |  MR | Is |  to |  detect |  abnormalities |  and |  identify |  potential |  health |  risks | .

 | AI |  is |  also |  being |  used |  in |  person

## `invoke` vs `stream`

Both send your input to the model. The difference is **how you get the answer back**.

### `invoke`
- Waits for the **full response**, then returns it all at once.
- You get one complete result.

```python
result = model.invoke("What is LangChain?")
print(result.content)
```

- Best for: short answers, when you need the whole result before moving on.

### `stream`
- Returns the answer **piece by piece** (token by token) as it is generated.
- You loop over the chunks.

```python
for chunk in model.stream("What is LangChain?"):
    print(chunk.content, end="", flush=True)
```

- Best for: chat UIs and long answers — feels live, like typing.

### Quick table

| Feature | `invoke` | `stream` |
|---------|----------|----------|
| Output | Full answer at once | Small chunks, live |
| Return type | One result object | A generator (loop over it) |
| Wait time | Wait for whole reply | See words as they come |
| Best for | Short/simple calls | Long replies, chat feel |

### Note
- `invoke` → use `result.content`
- `stream` → use `chunk.content` inside a loop

----------
----------
---------

## `batch` in LangChain

`batch` runs **many inputs at once**, instead of calling the model one by one.

### How it works
- You give it a **list** of inputs.
- It processes them **in parallel** (faster than a loop).
- It returns a **list of results** in the **same order**.

```python
questions = [
    "What is LangChain?",
    "What is LangGraph?",
    "What is an embedding?",
]

results = model.batch(questions)

for r in results:
    print(r.content)
```

### Control how many run at once
Use `max_concurrency` to avoid overloading the gateway or hitting rate limits.

```python
results = model.batch(
    questions,
    config={"max_concurrency": 2},   # only 2 at a time
)
```

### invoke vs stream vs batch

| Method | Input | Output | Best for |
|--------|-------|--------|----------|
| `invoke` | 1 input | 1 full result | Single call |
| `stream` | 1 input | Live chunks | Chat, long replies |
| `batch` | list of inputs | list of results | Many inputs, fast |

### Note
- Results come back **in order** — first input = first result.
- Use `r.content` for each result.
- For the GE gateway, keep `max_concurrency` low to respect limits.

In [ ]:
responses_of_batch = model.batch([
    "What is Langchain?",
    "What is the difference between Langchain and LLM?",
    "Why we use Langchain?",
    "Where is pacific ocean located?"
],
config={
    'max_concurrency': 2,  # Limit the number of concurrent requests to LLM 
})
for response in responses_of_batch:
    print(response)

content="Langchain is an open-source framework for building and integrating large language models (LLMs) with other AI systems. It was developed by Meta AI and is designed to make it easier to work with LLMs in a variety of applications, such as natural language processing (NLP), computer vision, and more.\n\nLangchain provides a set of tools and APIs that allow developers to:\n\n1. Integrate LLMs with other AI systems, such as computer vision models or reinforcement learning agents.\n2. Use LLMs as a component in larger models, such as transformer-based architectures.\n3. Fine-tune pre-trained LLMs on specific tasks or datasets.\n\nLangchain's key features include:\n\n1. **Model Zoo**: A collection of pre-trained LLMs that can be easily integrated into applications.\n2. **APIs**: A set of APIs for interacting with LLMs, including text generation, question answering, and more.\n3. **Integration tools**: Tools for integrating LLMs with other AI systems, such as computer vision models or

## How to use Steam and Batch together

#### Option 1 — Loop over inputs, stream each one

In [56]:
questions = [
    "What is LangChain?",
    "What is LangGraph?",
    "What is an embedding?",
]

for q in questions:
    print(f"\n\nQ: {q}\n{'-'*40}")
    for chunk in model.stream(q):
        print(chunk.content, end="", flush=True)



Q: What is LangChain?
----------------------------------------
LangChain is an open-source, decentralized platform that enables the creation of blockchain-based data pipelines and workflows. It allows developers to build custom data processing and storage solutions on top of blockchain technology.

LangChain provides a set of tools and libraries that enable users to create, manage, and execute data pipelines, which are essentially workflows that process and transform data in a specific order. These pipelines can be used for a wide range of applications, such as data analytics, machine learning, and IoT (Internet of Things) use cases.

The key features of LangChain include:

1. Data pipeline creation: Users can create custom data pipelines using a visual interface or through code.
2. Blockchain-based storage: LangChain provides a blockchain-based storage solution for data, which ensures that data is secure, decentralized, and tamper-proof.
3. Smart contract integration: LangChain allo

#### Option 2 — batch for speed, then print results

- Use batch to run everything in parallel (fast), then display the finished answers. This isn't "live" streaming, but it's the fastest for many inputs.

In [52]:
results = model.batch(questions, config={"max_concurrency": 2})

for q, r in zip(questions, results):
    print(f"\nQ: {q}\n{'-'*40}")
    print(r.content)


Q: What is LangChain?
----------------------------------------
LangChain is an open-source, decentralized platform that enables the creation of blockchain-based data pipelines and workflows. It allows developers to build custom data processing and storage solutions on top of blockchain technology.

LangChain provides a set of tools and libraries that enable users to create, manage, and execute data pipelines, which are essentially workflows that process and transform data in a specific order. These pipelines can be used for a wide range of applications, such as data analytics, machine learning, and IoT (Internet of Things) use cases.

The key features of LangChain include:

1. Data pipeline creation: Users can create custom data pipelines using a visual interface or through code.
2. Blockchain-based storage: LangChain provides a blockchain-based storage solution for data, which ensures that data is secure, decentralized, and tamper-proof.
3. Smart contract integration: LangChain allow

In [55]:
import asyncio

async def stream_one(q):
    print(f"\nStarting: {q}")
    text = ""
    async for chunk in model.astream(q):
        text += chunk.content
    print(f"\nDone: {q}\n{text}\n{'-'*40}")
    return text

async def run_all(questions):
    tasks = [stream_one(q) for q in questions]
    return await asyncio.gather(*tasks)

# In a Jupyter notebook:
results = await run_all(questions)



Starting: What is LangChain?

Starting: What is LangGraph?

Starting: What is an embedding?

Done: What is LangGraph?
LangGraph is an open-source, graph-based data structure and algorithm library for Rust programming language. It provides a simple and efficient way to work with graphs in Rust.

LangGraph allows you to create, manipulate, and query graphs using a variety of data structures such as adjacency lists, edge lists, and adjacency matrices. It also provides various algorithms for graph traversal, such as breadth-first search (BFS), depth-first search (DFS), and Dijkstra's algorithm.

Some key features of LangGraph include:

1. Efficient graph representation: LangGraph uses an efficient in-memory representation of graphs, making it suitable for large-scale graph processing.
2. Simple API: The library provides a simple and intuitive API for working with graphs, making it easy to learn and use.
3. Extensive algorithm support: LangGraph includes a wide range of algorithms for grap